# Gradient Boosting Trees

GBT is an incremental learning method based on gradient descent that gradually improves the model’s performance by iteratively building multiple decision trees. The core idea of GBT is to utilize gradient descent to optimize the model’s loss function, thereby step by step enhancing the model’s performance.

The training process of GBT is as follows:

First, an initial model is defined, which is usually a single decision tree.
Next, the model’s loss function is computed, typically by summing the differences between predicted and actual values.
Then, the gradient of the loss function is calculated, indicating the direction for improving the model in its current state.
Finally, the model is updated according to the gradient, usually through gradient descent.


In [21]:
import os

# Install the specified wheel file using pip
os.system("pip install rtdl_num_embeddings-0.0.11-py3-none-any.whl")

# Rename the file 'onebap.bin' to 'onebap.so'
os.system("mv onebap.bin onebap.so")

# Import necessary libraries
import pickle  # For serializing and deserializing Python objects
import torch  # PyTorch for deep learning
import torch.nn as nn  # Neural network modules
import torch.nn.functional as F  # Functions for neural networks
import torch.optim  # Optimizers for training
from torch.utils.data import Dataset, DataLoader, TensorDataset  # Data handling utilities
from sklearn.model_selection import train_test_split  # Splitting data into training and testing sets

In [22]:
from sklearn.metrics import r2_score  # For calculating R-squared score
import pandas as pd  # Data manipulation and analysis
import math  # Mathematical functions
import numpy as np  # Numerical computations
from tqdm import tqdm  # Progress bar for loops
import polars as pl  # Alternative to pandas for dataframes
from collections import OrderedDict  # Ordered dictionary for maintaining insertion order
import sys  # System-specific parameters and functions
# from tabm_reference import Model, make_parameter_groups  # Custom model and utility functions
import warnings  # For suppressing warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

In [24]:

import joblib  # For saving and loading Python objects
import gc
import warnings

In [37]:
# Read the Parquet file
data = pl.read_parquet('processed_data/Processed_data_1695.parquet')

# Add a 'row_id' column, starting from 0 and sequentially incrementing
data = data.with_row_count("row_id", offset=0)

# Add lagged columns for each responder (responder_{idx}_lag_1)
for idx in range(9):
    lag_col_name = f"responder_{idx}_lag_1"  # Name of the lagged column
    original_col_name = f"responder_{idx}"  # Name of the original column
    # Create the lagged column by shifting the original column by 1 row
    data = data.with_columns(pl.col(original_col_name).shift(1).alias(lag_col_name))

# Print the data after adding 'row_id' and lagged columns
print("\nData after adding 'row_id' and lagged columns:")


Data after adding 'row_id' and lagged columns:


In [25]:
data

row_id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,…,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_0,responder_1,responder_2,responder_3,responder_4,responder_5,responder_6,responder_7,responder_8,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
u32,i16,i16,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,1695,0,0,3.373552,2.776059,1.035769,1.790385,1.915069,1.765592,-0.105391,-0.16963,-0.345125,0.03763,11.0,7.0,76.0,-1.029333,0.271748,-0.640703,-0.688929,-0.763758,-0.664928,-1.701001,-1.246213,1.171051,-0.151066,1.780195,0.592178,2.011824,1.034978,1.035668,1.118305,0.616451,-0.932135,-1.078001,-0.154166,…,-0.019777,1.419774,-0.401114,-0.292407,-0.196831,-1.569348,-2.423477,-0.725855,0.203034,-0.451438,-0.841593,0.31118,-0.575253,0.171726,0.140167,1.225793,0.929635,0.067034,0.06966,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505,null,null,null,null,null,null,null,null,null
1,1695,0,1,2.802384,1.901147,1.029203,2.146977,2.811388,1.423654,-0.105936,-0.173366,-0.382062,0.045153,11.0,7.0,76.0,-1.263074,0.253009,-0.599086,-0.688929,-0.633995,-0.664928,-1.906501,-0.91079,0.886705,0.012082,1.116906,0.85608,1.12455,0.723127,-1.920028,-0.053758,1.010441,-0.559132,-0.940507,0.010087,…,-0.079671,1.419774,-0.49611,-0.439725,-0.442559,-1.185965,-1.933517,-1.124566,0.131424,-0.604832,-0.760508,0.142326,-0.745543,0.171726,0.140167,-0.226083,-0.213198,-0.206452,-0.373929,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505
2,1695,0,2,2.506616,2.801315,1.376992,2.313733,2.135918,1.765473,-0.173767,-0.261385,-0.473618,0.018804,81.0,2.0,59.0,-0.99006,0.873093,-0.564999,-0.688929,-0.389982,-0.664928,-1.932154,-0.971507,0.023837,-0.145004,0.877403,0.370831,0.23665,-0.146222,0.580706,0.660125,0.125094,-0.627553,-0.784622,-0.183313,…,1.418136,1.419774,-0.363737,-0.314952,-0.355761,-1.836319,-1.781683,-0.99552,-0.030046,-0.476795,-0.888185,1.50716,-0.264173,0.171726,0.140167,1.517169,1.503251,0.094979,0.109544,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547
3,1695,0,3,1.868808,2.492323,1.293729,2.051276,2.601039,2.115296,-0.137015,-0.15458,-0.39843,0.058264,4.0,3.0,11.0,-1.047425,1.116742,-0.303803,-0.688929,-0.510801,-0.664928,-1.360154,-1.818448,-0.282608,-0.068771,0.591393,0.9088,0.051669,-0.261807,-0.01116,-0.169174,-0.219266,-0.642355,-0.530943,-0.037706,…,-0.0473,1.419774,-0.315971,-0.243883,-0.276656,-2.015491,-2.266636,-1.011813,0.711239,-0.346282,-1.06627,0.20662,-0.518951,0.171726,0.140167,0.425632,0.424689,-0.129259,-0.146268,0.047888,-0.01461,-0.334644,-0.0426,-0.569183,-0.244618,-0.055909,-0.570931,-0.120065,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755
4,1695,0,4,2.826087,2.754668,0.981692,2.405406,2.504181,2.060546,-0.067486,-0.153702,-0.205151,0.021181,15.0,1.0,9.0,-0.840723,1.097859,-0.419687,-0.688929,-0.834465,-0.664928,-1.323989,-1.13767,-1.158483,0.721674,0.600382,0.

In [26]:
weights = data['weight'].to_numpy().tolist()
train = data.drop(['weight'])

del data
gc.collect()

11326

### Get X,y 

In [27]:
cols=[f'feature_0{i}' if i<10 else f'feature_{i}' for i in range(79)]
X=train.select(cols).fill_null(3).to_numpy()
y=train.select('responder_6').to_numpy().flatten()
del train
gc.collect()

0

## Train test split

In [38]:
split_ratio = 0.2  # 20% for test data, 80% for training  
split = int(len(X) * split_ratio)  # Calculate the split index dynamically 
train_X,train_y,test_X,test_y,train_weight,test_weight=X[:-split],y[:-split],X[-split:],y[-split:],weights[:-split],weights[-split:]
print(f"train_X.shape:{train_X.shape},test_X.shape:{test_X.shape}")

train_X.shape:(120032, 79),test_X.shape:(30008, 79)


## GBT Model

In [29]:
from sklearn.ensemble import GradientBoostingRegressor  

### Fit 

In [30]:
def custom_metric(y_true,y_pred,weight):
    weighted_r2=1-(np.sum(weight*(y_true-y_pred)**2)/np.sum(weight*y_true**2))
    return weighted_r2

In [31]:
import numpy as np  
import torch  
import torch.nn as nn  
import torch.optim as optim  
import polars as pl  
import copy 

This code implements a Gradient Boosting Trees (GBT) model and uses PyTorch for training and prediction. The GBTModel class inherits from the 'nn. Module', which defines multiple decision trees in the initialization method, each consisting of two layers of fully connected neural networks. In the forward propagation method, the initial output is all zero, and then the final output is obtained by multiplying the prediction result for each tree by the learning rate. The 'train_GBT' function is used to train the GBT model, using the Mean Square Error (MSE) as the loss function and the Stochastic Gradient Descent (SGD) optimizer. During training, the model monitors the loss value at the end of each epoch and stops training early if there is no improvement after multiple epochs in a row. The 'predict_GBT' function is used for model prediction, which converts the test data into a Tensor and propagates it forward to return the prediction result. Finally, the code instantiates a GBT model and trains it.

In [39]:
class GBTModel(nn.Module):  
    def __init__(self, n_estimators, n_features, learning_rate=0.1):  
        super(GBTModel, self).__init__()  
        self.n_estimators = n_estimators  
        self.learning_rate = learning_rate  
        
        self.trees = nn.ModuleList()  
        for _ in range(n_estimators):  
            tree = nn.Sequential(  
                nn.Linear(n_features, 16),  
                nn.ReLU(),  
                nn.Linear(16, 1)  
            )  
            self.trees.append(tree)  

    def forward(self, x):  
        # 输出初始值: 全0  
        output = torch.zeros(x.size(0), 1, device=x.device)  
        # 把 learning_rate 用来缩放每棵树的预测
        for tree in self.trees:  
            output += self.learning_rate * tree(x)  
        return output  

import copy  

def train_GBT(  
    model, x_train, y_train,   
    n_epochs, batch_size, device,   
    patience=3  # 连续多少个 epoch 无改进则停止  
):  
    criterion = nn.MSELoss()  
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)  

    model.to(device)  

    best_loss = float('inf')  
    epochs_no_improve = 0  
    best_model_state = None  

    for epoch in range(n_epochs):  
        permutation = np.random.permutation(len(x_train))  

        for i in range(0, len(x_train), batch_size):  
            indices = permutation[i:i+batch_size]  
            inputs = x_train[indices]  
            targets = y_train[indices]  

            inputs = torch.from_numpy(inputs).float().to(device)  
            targets = torch.from_numpy(targets).float().to(device).view(-1, 1)  

            optimizer.zero_grad()  
            outputs = model(inputs)  
            loss = criterion(outputs, targets)  
            loss.backward()  
            optimizer.step()  

        # 在每个 epoch 末进行监控  
        if loss.item() < best_loss:  
            best_loss = loss.item()  
            epochs_no_improve = 0  
            best_model_state = copy.deepcopy(model.state_dict())  
        else:  
            epochs_no_improve += 1  

        if (epoch + 1) % 10 == 0:  
            print(f"Epoch {epoch+1}/{n_epochs}, Loss = {loss.item():.6f}")  

        # 若连续多个 epoch 无改进，则提前停止  
        if epochs_no_improve >= patience:  
            print(f"Early stopping at epoch {epoch+1}")  
            break  

    # 加载在训练期间表现最好的参数  
    if best_model_state is not None:  
        model.load_state_dict(best_model_state)  

def predict_GBT(model, x_test, device):  
    model.eval()  
    with torch.no_grad():  
        x_test_tensor = torch.from_numpy(x_test).float().to(device)  
        outputs = model(x_test_tensor)  
    return outputs.cpu().numpy().flatten()  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
print(f"Using device: {device}")  

n_estimators = 10        # 树的数量 
n_features = 79          # 特征维度  
learning_rate = 0.1  
n_epochs = 10  
batch_size = 64  

model = GBTModel(  
    n_estimators=n_estimators,  
    n_features=n_features,  
    learning_rate=learning_rate  
)  

train_GBT(model, train_X, train_y, n_epochs, batch_size, device=device)  

Using device: cuda
Early stopping at epoch 7


In [40]:
# 训练完后可进行预测  
train_pred = predict_GBT(model, train_X, device=device)  
test_pred = predict_GBT(model, test_X, device=device)  

print(f"train weighted_r2: {custom_metric(train_y, train_pred, weight=train_weight)}")  
print(f"test weighted_r2:  {custom_metric(test_y, test_pred, weight=test_weight)}")

train weighted_r2: 0.001437804782996266
test weighted_r2:  -0.06948098315060203
